# Feedforward networks, end to end

A two-layer network on the same dataset as the other notebooks, fitted twice: once on scaled inputs and once on raw ones. Everything else — architecture, seed, stopping rule — is held identical, so the difference between the two lines is attributable to scaling alone.

`early_stopping=True` holds back 10% of the training rows and halts when that slice stops improving for 20 epochs. Without it the network keeps descending into the training set long after it has stopped learning anything general.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Same dataset as the other notebooks, so the numbers are comparable
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)


def build(**kwargs):
    # Two hidden layers. early_stopping holds out 10% to decide when to quit.
    return MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2000,
                         early_stopping=True, n_iter_no_change=20,
                         random_state=42, **kwargs)


# 2. Scaled: the network sees every feature on the same footing
scaled = make_pipeline(StandardScaler(), build())
scaled.fit(X_train, y_train)
net = scaled[-1]
print(f"scaled    acc={accuracy_score(y_test, scaled.predict(X_test)):.4f}  "
      f"epochs={net.n_iter_:3}  final loss={net.loss_:.4f}")

# 3. Unscaled: same architecture, same seed, raw inputs
raw = build()
raw.fit(X_train, y_train)
print(f"unscaled  acc={accuracy_score(y_test, raw.predict(X_test)):.4f}  "
      f"epochs={raw.n_iter_:3}  final loss={raw.loss_:.4f}")

# 4. Gradient descent leaves a trace: the loss curve it walked down
curve = scaled[-1].loss_curve_
print("\nscaled loss curve (every 10th epoch):")
print("  " + "  ".join(f"{v:.3f}" for v in curve[::10]))

scaled    acc=0.9649  epochs= 39  final loss=0.0948
unscaled  acc=0.9035  epochs= 60  final loss=0.1983

scaled loss curve (every 10th epoch):
  0.925  0.342  0.176  0.118

## What the output is telling you

- **Scaling is worth about six points of accuracy here** — 0.9649 against 0.9035 — with nothing else changed. This is the most consequential line in the notebook.
- **It also converged faster**: 39 epochs against 60. Raw features arrive on wildly different scales, so the loss surface is stretched into a narrow valley and gradient descent has to zig-zag down it. Standardising makes it round.
- **The loss curve is the descent itself**: 0.925 → 0.342 → 0.176 → 0.118. Flat from the start usually means the learning rate is too small or the inputs are unscaled; violent jumps mean it is too large.
- **0.9649 still loses to logistic regression's 0.9737.** A network with thousands of parameters had no advantage over a linear boundary on 455 rows, because there was no non-linear structure to find.

## When to reach for this

Neural networks earn their keep where the features are not already meaningful columns: images, audio, text, sequences — anything where the representation has to be learned rather than engineered. They also scale with data in a way the other models do not.

On tabular data of this size they are the wrong tool, and this notebook is the evidence. Gradient boosting beats them on tabular problems often enough that it should be your default there.

## Extend this notebook

- Swap `hidden_layer_sizes` to `(2,)` and then `(256, 128, 64)`. Neither should help, which is the point.
- Set `early_stopping=False` and compare `loss_` against test accuracy — the gap is overfitting.
- Adjust `alpha`, the L2 penalty, and watch the same bias-variance trade-off the ridge lesson describes.
- For real work, reach for PyTorch or Keras: `MLPClassifier` is the right teaching tool but has no GPU support, no custom losses, no control over the training loop.